# A miniature DeepSeek-V3

*DeepSeek-V3 technical report. Notebook 8 of 8: putting the pieces together.*

**TL;DR.** Notebooks 1 to 7 each built one idea in isolation. Here we put the architecture ideas into **one small language model** and check that they work together: Multi-head Latent Attention (notebook 1), DeepSeekMoE with shared and routed experts and node-limited routing (notebook 2), bias-based load balancing plus a tiny sequence-wise loss (notebook 3), and a multi-token-prediction module (notebook 4). We verify that the whole model decodes correctly with its compressed cache, train it next to an ordinary dense transformer, measure expert balance, and run lossless speculative decoding with its own MTP module. FP8 training (5), DualPipe (6) and YaRN (7) are *training-system* and *context-extension* techniques; they are not part of this model, and the table below says so.

| paper component | notebook | in this model? |
|---|---|---|
| Multi-head Latent Attention | 1 | yes |
| DeepSeekMoE: shared + routed experts, sigmoid gates | 2 | yes |
| Node-limited routing | 2 | yes (4 groups, at most 2 per token) |
| Auxiliary-loss-free balancing + sequence-wise loss | 3 | yes |
| Multi-token prediction (depth 1) | 4 | yes |
| FP8 mixed precision | 5 | no: emulation is too slow to train with here |
| DualPipe | 6 | no: needs many GPUs |
| YaRN | 7 | no: a short window here |

In [1]:
import sys, math, time
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))             # the notebook lives next to dsv3kit.py
import dsv3kit as K                             # tested building blocks; this notebook re-shows the code it uses

torch.manual_seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
C = dict(blue="#1f78b4", orange="#d95f02", green="#1b9e77", purple="#7570b3", red="#c0392b", grey="#777777")

PASS, FAIL = [], []
def check(name, ok, detail=""):
    (PASS if ok else FAIL).append(name)
    print(("PASS" if ok else "FAIL"), "-", name, detail)

print("torch", torch.__version__)

torch 2.14.1


## 1. The architecture

A DeepSeek-V3 block is: RMSNorm, attention (MLA), residual, RMSNorm, feed-forward (dense in the first layers, DeepSeekMoE after), residual. On top of the stack sit the output head (shared with the embedding) and one MTP module. We set the same structure at a small size:

| | DeepSeek-V3 | our mini model |
|---|---|---|
| layers (first dense) | 61 (3) | 4 (1) |
| hidden size | 7168 | 96 |
| attention | MLA, 128 heads, $d_c=512$, $d^R_h=64$ | MLA, 4 heads, $d_c=24$, $d^R_h=12$ |
| routed experts / active | 256 / 8 | 8 / 2 |
| shared experts | 1 | 1 |
| node-limited | 4 of 8 nodes | 2 of 4 groups |
| MTP depth | 1 | 1 |

The "baseline" for comparison is an ordinary transformer with standard multi-head attention and a dense feed-forward network, with about the same number of parameters *active per token*.

In [2]:
data = K.CharData()
mini = K.LMConfig(vocab=data.vocab, d=96, n_layers=4, n_heads=4, first_dense=1, d_ff=192,
                  attn="mla", d_c=24, d_cq=36, d_r=12,
                  ffn="moe", n_shared=1, n_routed=8, top_k=2, moe_ff=48, n_groups=4, topk_groups=2,
                  mtp_depth=1, mtp_lambda=0.3)
dense = K.LMConfig(vocab=data.vocab, d=96, n_layers=4, n_heads=4, d_ff=192, attn="mha", ffn="dense", mtp_depth=0)

def param_report(model):
    total = sum(p.numel() for p in model.parameters())
    mtp = sum(p.numel() for p in model.mtp.parameters())
    inactive = sum((m.n_routed - m.top_k) * sum(p.numel() for p in m.routed[0].parameters()) for m in model.moes())
    return total, total - inactive, mtp

m_mini, m_dense = K.TinyLM(mini), K.TinyLM(dense)
for name, m in [("dense baseline", m_dense), ("mini-V3", m_mini)]:
    tot, act, mtp = param_report(m)
    print(f"{name:15s} total {tot/1e3:7.1f}K parameters;  used per token {act/1e3:7.1f}K ({100 * act / tot:.0f}%);  MTP module {mtp/1e3:.1f}K (discarded at inference)")
print()
print(m_mini.blocks[0].attn.__class__.__name__, "+", m_mini.blocks[0].ffn.__class__.__name__, " (layer 0: dense FFN, as in the paper)")
print(m_mini.blocks[1].attn.__class__.__name__, "+", m_mini.blocks[1].ffn.__class__.__name__, " (layers 1-3: MoE)")
moe1 = m_mini.moes()[0]
print(f"MoE layer: {len(moe1.shared)} shared + {moe1.n_routed} routed experts, top-{moe1.top_k}, {moe1.n_groups} groups, at most {moe1.topk_groups} per token")
kv_mha, kv_mla = 2 * 4 * 24, 24 + 12
print(f"KV cache per token per layer: baseline {kv_mha} numbers, mini-V3 {kv_mla} numbers  ({kv_mha / kv_mla:.1f}x smaller)")
check("mini-V3 uses MLA and MoE in the right layers", m_mini.blocks[0].ffn.__class__.__name__ == "DenseFFN" and all(isinstance(b.ffn, K.DeepSeekMoE) for b in m_mini.blocks[1:]) and isinstance(m_mini.blocks[1].attn, K.MLA))
check("the MoE model has more parameters than it uses per token", param_report(m_mini)[0] > param_report(m_mini)[1])

dense baseline  total   378.7K parameters;  used per token   378.7K (100%);  MTP module 0.0K (discarded at inference)
mini-V3         total   644.9K parameters;  used per token   396.1K (61%);  MTP module 100.1K (discarded at inference)

MLA + DenseFFN  (layer 0: dense FFN, as in the paper)
MLA + DeepSeekMoE  (layers 1-3: MoE)
MoE layer: 1 shared + 8 routed experts, top-2, 4 groups, at most 2 per token
KV cache per token per layer: baseline 192 numbers, mini-V3 36 numbers  (5.3x smaller)
PASS - mini-V3 uses MLA and MoE in the right layers 
PASS - the MoE model has more parameters than it uses per token 


## 2. Does the whole model decode correctly with the compressed cache?

Notebook 1 verified one MLA layer. Here we verify the **assembled** model: generate token by token, giving each layer only its small cache (the latent and the RoPE key), and require the logits to equal those of a normal full-sequence forward pass. This exercises MLA's absorbed path, the MoE routing token by token, the residual stream and the output head all together.

In [3]:
@torch.no_grad()
def forward_cached(model, tokens):
    """Process tokens one at a time; each layer keeps only (latent, rope-key). Returns the logits for every position."""
    caches = [None] * len(model.blocks)
    outs = []
    for t in range(tokens.shape[1]):
        x = model.emb(tokens[:, t:t + 1])
        for li, blk in enumerate(model.blocks):
            a, caches[li] = blk.attn.forward_absorbed(blk.n1(x), caches[li])
            x = x + a
            x = x + blk.ffn(blk.n2(x))
        outs.append(model.head(x))
    return torch.cat(outs, 1), caches

torch.manual_seed(0)
m_mini.eval()
toks = torch.randint(0, data.vocab, (2, 24))
with torch.no_grad():
    full = m_mini(toks)
cached, caches = forward_cached(m_mini, toks)
print("max |full forward - cached decoding| over all positions and layers:", float((full - cached).abs().max()))
elems = sum(c[0].numel() + c[1].numel() for c in caches) / (2 * 24)
print(f"cache holds {elems:.0f} numbers per token over {len(caches)} layers = {elems / len(caches):.0f} per layer")
check("assembled model: cached decoding equals the full forward pass", torch.allclose(full, cached, atol=1e-4))
check("cache is d_c + d_r numbers per token per layer", abs(elems / len(caches) - 36) < 1e-9)

max |full forward - cached decoding| over all positions and layers: 4.76837158203125e-07
cache holds 144 numbers per token over 4 layers = 36 per layer
PASS - assembled model: cached decoding equals the full forward pass 
PASS - cache is d_c + d_r numbers per token per layer 


## 3. Train it next to a dense baseline

Four runs, same data and seeds, 300 steps of 48 sequences of 64 characters:

- **dense baseline**: MHA + dense FFN, no MTP;
- **mini-V3**: everything on, $\gamma = 0.01$ for the bias (larger than the paper's 0.001 because the run is so short) and a sequence-wise loss with $\alpha=0.001$;
- **mini-V3 without MTP**;
- **mini-V3 with no balancing** (no bias update, no sequence loss).

All report the **main head's** loss: MTP only helps the model, it is discarded for ordinary use. We use two seeds for the first three and one for the last. The laptop is shared, so the budget is small; the largest seed spread in the table is the yardstick for differences.

In [4]:
variants = {
    "dense baseline": dict(cfg=dense, kw={}),
    "mini-V3": dict(cfg=mini, kw=dict(gamma=0.01, seq_alpha=0.001)),
    "mini-V3, no MTP": dict(cfg=K.LMConfig(**{**mini.__dict__, "mtp_depth": 0}), kw=dict(gamma=0.01, seq_alpha=0.001)),
    "mini-V3, no balancing": dict(cfg=mini, kw={}),
}
n_seeds = {"dense baseline": 2, "mini-V3": 2, "mini-V3, no MTP": 2, "mini-V3, no balancing": 1}

maxvio = lambda load: float((load.max() - load.mean()) / load.mean())
@torch.no_grad()
def batch_maxvio(model):
    model.eval()
    K.lm_loss(model, data.batch("val", 64, 64, torch.Generator().manual_seed(7)))
    v = [maxvio(m.last["load"].float()) for m in model.moes()]
    model.train()
    return float(np.mean(v)) if v else float("nan")

results8, trained = {}, {}
t0 = time.time()
for name, v in variants.items():
    rows = []
    for seed in range(n_seeds[name]):
        model, _ = K.train_lm(v["cfg"], data, steps=300, B=48, T=64, lr=4e-3, seed=seed, **v["kw"])
        ev = K.evaluate(model, data, T=64, n_batches=10)
        rows.append((ev["main"], batch_maxvio(model)))
        trained[(name, seed)] = model
    results8[name] = np.array(rows)
    r = results8[name]
    print(f"{name:24s} main-head val loss {r[:, 0].mean():.3f} +- {r[:, 0].std():.3f}   expert-load MaxVio {np.nanmean(r[:, 1]):.2f}   ({len(r)} seed{'s' if len(r) > 1 else ''})")
print("time %.0f s" % (time.time() - t0))

/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_50775/3831324900.py:29: RuntimeWarning: Mean of empty slice
  print(f"{name:24s} main-head val loss {r[:, 0].mean():.3f} +- {r[:, 0].std():.3f}   expert-load MaxVio {np.nanmean(r[:, 1]):.2f}   ({len(r)} seed{'s' if len(r) > 1 else ''})")


dense baseline           main-head val loss 2.483 +- 0.006   expert-load MaxVio nan   (2 seeds)


mini-V3                  main-head val loss 2.484 +- 0.035   expert-load MaxVio 0.28   (2 seeds)


mini-V3, no MTP          main-head val loss 2.429 +- 0.018   expert-load MaxVio 0.21   (2 seeds)


mini-V3, no balancing    main-head val loss 2.548 +- 0.000   expert-load MaxVio 1.54   (1 seed)
time 1128 s


In [5]:
r = {k: v.mean(0) for k, v in results8.items()}
spread = max(v[:, 0].std() for v in results8.values() if len(v) > 1)
print(f"mini-V3 minus dense baseline : {r['mini-V3'][0] - r['dense baseline'][0]:+.3f} nats")
print(f"MTP effect (full minus no-MTP): {r['mini-V3'][0] - r['mini-V3, no MTP'][0]:+.3f} nats")
print(f"balancing effect on balance   : MaxVio {r['mini-V3, no balancing'][1]:.2f} without -> {r['mini-V3'][1]:.2f} with")
print(f"largest seed spread           : {spread:.3f} nats")
check("all variants learn", all(v[:, 0].mean() < math.log(data.vocab) - 1.0 for v in results8.values()))
check("the balancing bias lowers expert-load imbalance", r["mini-V3"][1] < r["mini-V3, no balancing"][1], "(%.2f vs %.2f)" % (r["mini-V3"][1], r["mini-V3, no balancing"][1]))

mini-V3 minus dense baseline : +0.001 nats
MTP effect (full minus no-MTP): +0.055 nats
balancing effect on balance   : MaxVio 1.54 without -> 0.28 with
largest seed spread           : 0.035 nats
PASS - all variants learn 
PASS - the balancing bias lowers expert-load imbalance (0.28 vs 1.54)


How to read this: the model that has more parameters (mini-V3 holds about 1.6 times as many parameters as it uses per token) and a 5x smaller cache is compared with a dense model of the same compute. If the difference between them is within the printed seed spread, the honest conclusion is "no detectable difference at this scale". The one claim that does not depend on scale, because it is a mechanism rather than a quality claim, is the balancing line: with the bias on, expert load is more even.

## 4. Generate and verify lossless speculative decoding on the trained model

The trained mini-V3 includes its MTP module, so we can run the speculative decoder of notebook 4 on the complete model (MLA, MoE and all). The requirement is the same: the output must be **identical** to ordinary greedy decoding. (As in notebook 4, greedy output from a model this small loops on repeated words, which makes drafts easy to accept, so the acceptance rate printed below says nothing about V3's.)

In [6]:
@torch.no_grad()
def greedy(model, prompt, n_new):
    toks = list(prompt)
    for _ in range(n_new):
        toks.append(int(model(torch.tensor([toks]))[0, -1].argmax()))
    return toks

@torch.no_grad()
def mtp_draft(model, h, toks, p):
    mod = model.mtp[0]
    h1 = mod(h[:, :p + 1], model.emb(torch.tensor([toks[1:p + 2]])))
    return int(model.head(h1)[0, -1].argmax())

@torch.no_grad()
def speculative(model, prompt, n_new):
    committed = list(prompt)
    h = model.trunk(torch.tensor([committed]))
    x = int(model.head(h)[0, -1].argmax())
    d = mtp_draft(model, h, committed + [x], len(committed) - 1)
    passes, acc, tried = 1, 0, 0
    while len(committed) < len(prompt) + n_new:
        L = len(committed)
        h = model.trunk(torch.tensor([committed + [x, d]])); passes += 1
        logits = model.head(h)[0]
        y = int(logits[L].argmax()); tried += 1
        if y == d:
            acc += 1; committed += [x, d]
            x = int(logits[L + 1].argmax()); d = mtp_draft(model, h, committed + [x], L + 1)
        else:
            committed += [x]; x = y; d = mtp_draft(model, h, committed + [x], L)
    return committed[:len(prompt) + n_new], passes, acc, tried

model = trained[("mini-V3", 0)].eval()
prompts = [data.val[i:i + 32].tolist() for i in (0, 7000, 21000)]
same, tot_pass, acc, tried = [], 0, 0, 0
for pr in prompts:
    g = greedy(model, pr, 60)
    s, sp, a_, t_ = speculative(model, pr, 60)
    same.append(g == s); tot_pass += sp; acc += a_; tried += t_
print("speculative output identical to greedy:", same)
print(f"draft acceptance {100 * acc / tried:.0f}%;  tokens per forward pass {3 * 60 / tot_pass:.2f}")
print("prompt :", repr(data.decode(prompts[0])))
print("output :", repr(data.decode(g[32:])))
check("speculative decoding on the assembled model is lossless", all(same))

speculative output identical to greedy: [True, True, True]
draft acceptance 73%;  tokens per forward pass 1.67
prompt : 'age)\n    U->>R: run diffusion/te'
output : ' the pale the pale the pale the pale the palle that then tha'
PASS - speculative decoding on the assembled model is lossless 


## 5. What the series established, and what it did not

The table is deliberately blunt about the status of each claim.

| claim from the paper | what we did | status |
|---|---|---|
| MLA shrinks the KV cache 57x at V3 shapes | computed from the shapes; verified the layer, absorbed and incremental paths | **verified (arithmetic and mechanism)** |
| MLA matches MHA in quality | toy comparison, 2 seeds | **not reproduced**: MLA was slightly worse at toy scale |
| DeepSeekMoE: shared + fine-grained experts | built and verified against the equations; toy tests | **mechanism verified**; benefit at scale not tested |
| Node-limited routing caps nodes per token | verified the rule and the cap | **verified**; training effect not tested |
| Bias-based balancing works without an auxiliary loss | control loop and toy training | **mechanism verified**; toy showed lower error than a loss at equal balance |
| Aux-loss-free beats aux-loss in quality (Table 5) | toy | **not tested at scale** |
| MTP improves quality | toy, 2 seeds (notebooks 4 and 8) | **no benefit detected**: no effect in notebook 4; in notebook 8 the main head was 0.055 nats worse with MTP, about 1.5 seed-spreads |
| MTP enables lossless speculative decoding, ~1.8x | built it, checked identity | **lossless verified**; speed-up not measured on hardware |
| FP8 with fine-grained scaling trains stably | emulated GEMMs and a toy regression | **mechanism verified**; no speed; paper's 0.25% not reproduced |
| DualPipe reduces bubbles and overlaps communication | simulator, Table 2 formulas, real-tensor gradient check | **verified in simulation**; no GPUs |
| YaRN extends context | angle-level checks, toy evaluation | **mechanism verified**; no long-context quality test |

## What this notebook does not show
- **Any statement about DeepSeek-V3's quality.** The model here has well under a million parameters, trains for 300 steps on 460K characters, and is compared at two seeds. The things it can establish are that the components compose, that the assembled model's compressed-cache decoding is exact, that balancing balances, and that speculative decoding is lossless.
- **The efficiency claims.** Training cost, serving cost and communication overlap all depend on hardware at a scale this notebook does not touch.

In [7]:
print("passed:", len(PASS), "| failed:", len(FAIL))
assert not FAIL, FAIL

passed: 7 | failed: 0
